# AFS-VFM Phase 2B - Internal Signal Probing (v2)

Extracts internal model representations from DINOv2, CLIP, and DETR.

**GPU:** Kaggle `Settings > Accelerator > GPU T4 x2` | Colab `Runtime > T4 GPU`

**Time:** ~2-3 hours on T4 (300 images x 5 degradations x 20 frames x 3 models)

In [ ]:
# == Step 1: Check GPU ==
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")

In [ ]:
# == Step 2: Clone repo and install ==
!git clone https://github.com/AmanDevNet/AFS-VFM.git
%cd AFS-VFM
!pip install -q -r requirements.txt timm datasets huggingface_hub scipy

In [ ]:
# == Step 3: HuggingFace login ==
from huggingface_hub import login
login(token="PASTE_YOUR_HF_TOKEN_HERE")

In [ ]:
# == Step 4: Download datasets ==
!python -u download_dataset.py

In [ ]:
# == Step 5: Select images ==
import os, sys, random
import pandas as pd
sys.path.insert(0, 'src')

os.makedirs('results/phase2b/embeddings', exist_ok=True)
os.makedirs('results/phase2b/signals', exist_ok=True)

COCO_DIR = 'data/coco_val'
IMAGENET_DIR = 'data/imagenet_val'
SUBSET_SIZE = 300

all_images = []
for directory, label in [(COCO_DIR, 'coco'), (IMAGENET_DIR, 'imagenet')]:
    if os.path.isdir(directory):
        for f in sorted(os.listdir(directory)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append((os.path.join(directory, f), label))
print(f"Total available images: {len(all_images)}")

if os.path.exists('results/phase2b/image_list.txt'):
    with open('results/phase2b/image_list.txt') as f:
        selected = set(line.strip() for line in f if line.strip())
    probe_images = [(p, ds) for p, ds in all_images if os.path.basename(p) in selected]
    print(f"Using Phase 2A strategic subset: {len(probe_images)} images")
else:
    random.seed(42)
    probe_images = random.sample(all_images, min(SUBSET_SIZE, len(all_images)))
    print(f"Using random subset: {len(probe_images)} images")

print(f"Images to probe: {len(probe_images)}")
print(f"Total inferences: {len(probe_images) * 5 * 20 * 3:,}")

In [ ]:
# == Step 6: Load models ==
from models.model_loader import VisionModelLoader
from models.signal_extractor import SignalExtractor

loader = VisionModelLoader()
device = loader.device

dino_model, dino_proc = loader.load_dinov2()
clip_model, clip_proc = loader.load_clip()
detr_model, detr_proc = loader.load_detr()

extractor = SignalExtractor(device)
print("All models loaded.")

In [ ]:
# == Step 6.5: Quick sanity test (1 image, 1 frame) ==
from PIL import Image
test_path = probe_images[0][0]
test_img = Image.open(test_path).convert('RGB')
print(f"Testing with: {test_path}")

print("\n--- DINOv2 ---")
try:
    d = extractor.extract_dinov2(dino_model, dino_proc, test_img)
    print(f"  prediction: {d['prediction']}")
    print(f"  confidence: {d['top1_confidence']:.4f}")
    print(f"  embedding shape: {d['cls_embedding'].shape}")
    print(f"  PASS")
except Exception as e:
    print(f"  FAILED: {type(e).__name__}: {e}")

print("\n--- CLIP ---")
CLIP_LABELS = ['a dog','a cat','a car','a person','a bird','a tree','a chair','fruit',
               'a bicycle','an airplane','a clock','a coffee cup','a traffic scene',
               'a boat','a horse','a bottle','a bus','a truck','a train','a flower']
try:
    c = extractor.extract_clip(clip_model, clip_proc, test_img, CLIP_LABELS)
    print(f"  prediction: {c['prediction']}")
    print(f"  confidence: {c['top1_confidence']:.4f}")
    print(f"  embedding shape: {c['image_embedding'].shape}")
    print(f"  PASS")
except Exception as e:
    print(f"  FAILED: {type(e).__name__}: {e}")

print("\n--- DETR ---")
try:
    t = extractor.extract_detr(detr_model, detr_proc, test_img)
    print(f"  prediction: {t['prediction']}")
    print(f"  confidence: {t['top_detection_confidence']:.4f}")
    print(f"  PASS")
except Exception as e:
    print(f"  FAILED: {type(e).__name__}: {e}")

print("\n>> If all 3 say PASS, proceed to Step 7. If any FAILED, stop and report the error.")

In [ ]:
# == Step 7: Run Signal Probing ==
import csv, time, shutil, traceback
import numpy as np
from PIL import Image
from degradation import generate_degradation_sequence

DEGRADATION_TYPES = ['blur', 'occlusion', 'lighting', 'scale', 'viewpoint']
NUM_FRAMES = 20
CLIP_LABELS = ['a dog','a cat','a car','a bicycle','an airplane',
    'a clock','fruit','a person','a coffee cup','a traffic scene',
    'a bird','a boat','a horse','a chair','a bottle',
    'a bus','a truck','a train','a flower','a tree']

signals_csv_path = 'results/phase2b/signals/frame_signals.csv'

# Resume support
completed_images = set()
all_rows = []
if os.path.exists(signals_csv_path):
    existing = pd.read_csv(signals_csv_path)
    from collections import Counter
    img_counts = Counter(existing['image'])
    expected = len(DEGRADATION_TYPES) * NUM_FRAMES
    for img_name, count in img_counts.items():
        if count >= expected:
            completed_images.add(img_name)
    all_rows = existing.to_dict('records')
    print(f"Resuming: {len(completed_images)} images already done")

remaining = [(p, ds) for p, ds in probe_images if os.path.basename(p) not in completed_images]
total_images = len(probe_images)
error_counts = {'dinov2': 0, 'clip': 0, 'detr': 0}

print(f"Starting signal probing...")
print(f"  Total: {total_images}, Already done: {len(completed_images)}, Remaining: {len(remaining)}")
print(f"  Inferences remaining: {len(remaining) * 5 * 20 * 3:,}")

start_time = time.time()

for img_idx, (img_path, dataset_source) in enumerate(remaining, len(completed_images) + 1):
    img_file = os.path.basename(img_path)
    print(f"\n[{img_idx}/{total_images}] {img_file} ({dataset_source})")

    img_embeddings = {'dinov2': {}, 'clip': {}}

    for deg_type in DEGRADATION_TYPES:
        print(f"  {deg_type}", end=' ')

        try:
            frames = generate_degradation_sequence(img_path, deg_type, num_frames=NUM_FRAMES)
        except Exception as exc:
            print(f"SKIP: {exc}")
            continue

        baseline_dinov2 = None
        baseline_clip = None

        for frame_idx in range(NUM_FRAMES):
            frame = frames[frame_idx]
            severity_pct = int(frame_idx / (NUM_FRAMES - 1) * 100)

            if isinstance(frame, np.ndarray):
                frame_pil = Image.fromarray(frame)
            else:
                frame_pil = frame

            row = {
                'image': img_file,
                'dataset': dataset_source,
                'degradation': deg_type,
                'frame': frame_idx + 1,
                'severity_pct': severity_pct,
            }

            # ---- DINOv2 ----
            try:
                dino_sig = extractor.extract_dinov2(dino_model, dino_proc, frame_pil)
                row['dinov2_prediction'] = dino_sig['prediction']
                row['dinov2_top1_confidence'] = dino_sig['top1_confidence']
                row['dinov2_logit_margin'] = dino_sig['logit_margin']
                row['dinov2_attention_entropy'] = dino_sig['attention_entropy']
                if frame_idx == 0:
                    baseline_dinov2 = dino_sig
                    row['dinov2_cosine_sim'] = 1.0
                    row['dinov2_l2_distance'] = 0.0
                    row['dinov2_top5_overlap'] = 1.0
                else:
                    row['dinov2_cosine_sim'] = extractor.cosine_similarity(
                        baseline_dinov2['cls_embedding'], dino_sig['cls_embedding'])
                    row['dinov2_l2_distance'] = extractor.l2_distance(
                        baseline_dinov2['cls_embedding'], dino_sig['cls_embedding'])
                    row['dinov2_top5_overlap'] = extractor.top5_overlap(
                        baseline_dinov2['top5_labels'], dino_sig['top5_labels'])
                if deg_type == 'blur':
                    img_embeddings['dinov2'][frame_idx] = dino_sig['cls_embedding']
            except Exception as e:
                if error_counts['dinov2'] == 0:
                    print(f"\n  !! DINOv2 ERROR: {type(e).__name__}: {e}")
                    traceback.print_exc()
                error_counts['dinov2'] += 1
                row['dinov2_prediction'] = 'ERROR'
                for k in ['dinov2_top1_confidence','dinov2_logit_margin',
                          'dinov2_attention_entropy','dinov2_cosine_sim',
                          'dinov2_l2_distance','dinov2_top5_overlap']:
                    row[k] = float('nan')

            # ---- CLIP ----
            try:
                clip_sig = extractor.extract_clip(clip_model, clip_proc, frame_pil, CLIP_LABELS)
                row['clip_prediction'] = clip_sig['prediction']
                row['clip_top1_confidence'] = clip_sig['top1_confidence']
                row['clip_logit_margin'] = clip_sig['logit_margin']
                if frame_idx == 0:
                    baseline_clip = clip_sig
                    row['clip_cosine_sim'] = 1.0
                    row['clip_l2_distance'] = 0.0
                else:
                    row['clip_cosine_sim'] = extractor.cosine_similarity(
                        baseline_clip['image_embedding'], clip_sig['image_embedding'])
                    row['clip_l2_distance'] = extractor.l2_distance(
                        baseline_clip['image_embedding'], clip_sig['image_embedding'])
                if deg_type == 'blur':
                    img_embeddings['clip'][frame_idx] = clip_sig['image_embedding']
            except Exception as e:
                if error_counts['clip'] == 0:
                    print(f"\n  !! CLIP ERROR: {type(e).__name__}: {e}")
                    traceback.print_exc()
                error_counts['clip'] += 1
                row['clip_prediction'] = 'ERROR'
                for k in ['clip_top1_confidence','clip_logit_margin',
                          'clip_cosine_sim','clip_l2_distance']:
                    row[k] = float('nan')

            # ---- DETR ----
            try:
                detr_sig = extractor.extract_detr(detr_model, detr_proc, frame_pil)
                row['detr_prediction'] = detr_sig['prediction']
                row['detr_top_detection_confidence'] = detr_sig['top_detection_confidence']
                row['detr_objectness'] = detr_sig['objectness_score']
                row['detr_num_detections'] = detr_sig['num_detections_above_threshold']
            except Exception as e:
                if error_counts['detr'] == 0:
                    print(f"\n  !! DETR ERROR: {type(e).__name__}: {e}")
                    traceback.print_exc()
                error_counts['detr'] += 1
                row['detr_prediction'] = 'ERROR'
                for k in ['detr_top_detection_confidence','detr_objectness',
                          'detr_num_detections']:
                    row[k] = float('nan')

            all_rows.append(row)

        print("ok", end=' ')
    print()

    # Save embeddings
    emb_path = f"results/phase2b/embeddings/{img_file.replace('.', '_')}_embeddings.npz"
    np.savez_compressed(emb_path,
        **{f'dinov2_frame{k}': v for k, v in img_embeddings['dinov2'].items()},
        **{f'clip_frame{k}': v for k, v in img_embeddings['clip'].items()})

    # Crash-safe CSV save + backup to persistent dir
    df_out = pd.DataFrame(all_rows)
    df_out.to_csv(signals_csv_path, index=False)
    shutil.copy2(signals_csv_path, '/kaggle/working/frame_signals.csv')

    elapsed = time.time() - start_time
    imgs_done = img_idx - len(completed_images)
    rate = elapsed / imgs_done if imgs_done else 0
    eta = rate * (len(remaining) - imgs_done)
    print(f"  Saved [{img_idx}/{total_images}] ~{eta/60:.0f}min left  errors: d={error_counts['dinov2']} c={error_counts['clip']} t={error_counts['detr']}")

# ---- DONE ----
elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"SIGNAL PROBING COMPLETE")
print(f"  Images: {total_images}")
print(f"  Rows: {len(all_rows):,}")
print(f"  Time: {elapsed/3600:.1f}h")
print(f"  Errors: DINOv2={error_counts['dinov2']}, CLIP={error_counts['clip']}, DETR={error_counts['detr']}")
print(f"{'='*60}")

shutil.copy2(signals_csv_path, '/kaggle/working/frame_signals.csv')
shutil.make_archive('/kaggle/working/phase2b_embeddings', 'zip', 'results/phase2b/embeddings')

# Print summary
df = pd.read_csv(signals_csv_path)
print(f"\n--- SIGNAL SUMMARY ---")
print(f"DINOv2 confidence: mean={df['dinov2_top1_confidence'].mean():.4f}")
print(f"DINOv2 cosine sim: mean={df['dinov2_cosine_sim'].mean():.4f}")
print(f"CLIP confidence: mean={df['clip_top1_confidence'].mean():.4f}")
print(f"CLIP cosine sim: mean={df['clip_cosine_sim'].mean():.4f}")
print(f"DETR detection conf: mean={df['detr_top_detection_confidence'].mean():.4f}")
print(f"\n>> Download frame_signals.csv from Output tab")

In [ ]:
# == Step 8: Download ==
import base64
from IPython.display import HTML
with open('/kaggle/working/frame_signals.csv', 'rb') as f:
    data = base64.b64encode(f.read()).decode()
HTML(f'<a download="frame_signals.csv" href="data:text/csv;base64,{data}">Click to download frame_signals.csv</a>')